In [1]:
import requests
import datetime, time, logging, os, traceback
import pandas as pd
from numpy import nan

from urllib.parse import quote_plus
from sqlalchemy import create_engine
engine = create_engine('postgresql://zbrothers:%s@localhost:5434/zbrothers' % quote_plus('zbrothers@2026'))

In [28]:
API_TOKEN = os.environ['OLIST_API_TOKEN']

today = datetime.date.today()
# today = datetime.datetime(2026, 4, 15)
today_str = today.strftime('%d/%m/%Y')

In [25]:
def getInfosPesquisar(url: str, params: dict) -> pd.DataFrame:
    """
    Busca informações que são paginadas
    """
    res = requests.get(url, params=params)
    
    if res.json()['retorno']['status'] != 'OK':
        print(f"Falha ao obter quantidade de páginas: {res.json()['retorno']['status']}, possivelmente por sobrecarga na API.\nAguardando 90 segundos para tentar novamente...")
        time.sleep(90) 
        
        res = requests.get(url, params=params)
        
    num_pages = res.json()['retorno']['numero_paginas']
    print(f"Quantidade de páginas: {num_pages}")
    
    for i in range(1, num_pages + 1):
        print(f"Obtendo página {i} de {num_pages}...")
        params['pagina'] = i
        res = requests.get(url, params=params)
        
        if res.json()['retorno']['status'] != 'OK':
            print(f"Falha ao obter página {i}: {res.json()['retorno']['status']}, possivelmente por sobrecarga na API.\nAguardando 90 segundos para tentar novamente...")
            time.sleep(90) 
            
            res = requests.get(url, params=params)
        
        df = pd.DataFrame(res.json()['retorno']['separacoes'])
        
        if i == 1:
            df_final = df
        else:
            df_final = pd.concat([df_final, df], ignore_index=True)

    return df_final

In [ ]:
params={
        'token': API_TOKEN,
        'formato': 'json',
        'status': 1
}

res = requests.get('https://api.tiny.com.br/api2/separacao.pesquisa.php', 
                params={'token': API_TOKEN,
                        'formato': 'json',
                        'situacao': 1,
                        }
                )

In [20]:
res.json()['retorno']

{'status_processamento': '1',
 'status': 'Erro',
 'codigo_erro': 35,
 'erros': [{'erro': 'Não foi possível obter a separação'}]}

In [29]:
separacoes_aguardando_separacao = getInfosPesquisar(
    url='https://api.tiny.com.br/api2/separacao.pesquisa.php',
    params={
        'token': API_TOKEN,
        'formato': 'json',
        'situacao': 1,
        'dataInicial': {today_str}
    }
)

Quantidade de páginas: 1
Obtendo página 1 de 1...


In [31]:
separacoes_em_separacao = getInfosPesquisar(
    url='https://api.tiny.com.br/api2/separacao.pesquisa.php',
    params={
        'token': API_TOKEN,
        'formato': 'json',
        'situacao': 2,
        'dataInicial': {today_str}
    }
)

Quantidade de páginas: 1
Obtendo página 1 de 1...


In [32]:
separacoes_separadas = getInfosPesquisar(
    url='https://api.tiny.com.br/api2/separacao.pesquisa.php',
    params={
        'token': API_TOKEN,
        'formato': 'json',
        'situacao': 3,
        'dataInicial': {today_str}
    }
)

Quantidade de páginas: 8
Obtendo página 1 de 8...
Obtendo página 2 de 8...
Obtendo página 3 de 8...
Obtendo página 4 de 8...
Obtendo página 5 de 8...
Obtendo página 6 de 8...
Obtendo página 7 de 8...
Obtendo página 8 de 8...


In [ ]:
separacoes_embaladas = getInfosPesquisar(
    url='https://api.tiny.com.br/api2/separacao.pesquisa.php',
    params={
        'token': API_TOKEN,
        'formato': 'json',
        'situacao': 4,
        'dataInicial': {today_str}
    }
)

Falha ao obter quantidade de páginas: Erro, possivelmente por sobrecarga na API.
Aguardando 90 segundos para tentar novamente...


In [30]:
separacoes_aguardando_separacao

,id,idOrigem,objOrigem,situacao,dataCriacao,dataSeparacao,dataCheckout,destinatario,idContato,numero,dataEmissao,idFormaEnvio,numeroPedidoEcommerce,idOrigemVinc,objOrigemVinc,situacaoVenda
0,1075544010,1075544004,notafiscal,1,27/04/2026,None,None,Gustavo Ferreira,1031481038,188725,27/04/2026,1069030104,148,1075529847,venda,6
